# 08 — Grad-CAM (Figure 8)

**Tujuan.** Grad-CAM post-hoc pada Coordinate Attention dengan checkpoint eksperimen yang terverifikasi, serta pembacaan panel nyata APTOS2019 dan DDR validation grade 0–4 yang ditransfer dari server. Default sel real-image: APTOS2019 `balanced_v1/aptos_r16`, preprocessing V1, split validation. Figure 8 hlm. 91512–91513 tidak menetapkan layer target; `blocks.5.attention` adalah `ASSUMED`. Overlay sel lokal memakai piksel input model setelah crop/resize agar koordinat tepat; RGB asli pada panel transfer ditampilkan terpisah. Grad-CAM tidak masuk training loss. Tidak ada training atau final-test inference; output sintetis bukan citra fundus atau bukti lesi.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
import torch
from reproduction.cadvit.model import CADViTCandidate
from reproduction.cadvit.interpretability import gradcam, DEFAULT_TARGET_LAYER
torch.set_num_threads(1)
synthetic_model = CADViTCandidate().eval()
synthetic_result = gradcam(synthetic_model, torch.randn(1, 3, 64, 64))
print("SYNTHETIC_TEST target", DEFAULT_TARGET_LAYER,
      "heatmap", tuple(synthetic_result["heatmap"].shape),
      "range", (float(synthetic_result["heatmap"].min()), float(synthetic_result["heatmap"].max())))
print("PENDING real train/validation RGB + trusted checkpoint; no test image is opened")

SYNTHETIC_TEST target blocks.5.attention heatmap (64, 64) range (0.0, 0.0)
PENDING real train/validation RGB + trusted checkpoint; no test image is opened


In [ ]:
import csv
import hashlib
import numpy as np
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from reproduction.cadvit.interpretability import load_controlled_best, plot_gradcam
from reproduction.cadvit.visualization import guarded_non_test_image, save_campaign_figure
from reproduction.cadvit.experiments import artifact_paths, VARIANTS
from reproduction.cadvit.evaluation import file_sha256
from reproduction.cadvit.preprocessing import preprocess_fundus

if os.environ.get("CAD_VIT_RUN_GRADCAM") == "1":
    if ARTIFACT_ROOT is None:
        raise ValueError("Set CAD_VIT_ARTIFACT_ROOT to the verified external artifact root")
    campaign = os.environ.get("CAD_VIT_CAMPAIGN", "balanced_v1")
    variant = os.environ.get("CAD_VIT_VARIANT", "aptos_r16")
    source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "val")
    if (campaign, variant) != ("balanced_v1", "aptos_r16"):
        raise ValueError("Only the existing balanced APTOS V1 protocol is supported here; historical V2 and DDR recovery require separate provenance verification")
    if source_split not in {"train", "val"}:
        raise ValueError("Only train/val permitted")
    paths = artifact_paths(ARTIFACT_ROOT, VARIANTS[variant],
                           data_root=os.environ.get("CAD_VIT_DATA_ROOT"), campaign=campaign)
    dataset_root = Path(paths["dataset_root"]).resolve()
    manifest_path = Path(os.environ.get("CAD_VIT_AUDIT_MANIFEST", str(paths["manifest"])))
    for required in (manifest_path, paths["run_dir"] / "best.pt", paths["run_dir"] / "config.json"):
        if not required.is_file():
            raise FileNotFoundError(required)
    model, provenance = load_controlled_best(ARTIFACT_ROOT, campaign, variant, trust_checkpoint=True)
    if file_sha256(manifest_path) != provenance["manifest_sha256"]:
        raise ValueError("Audit manifest and checkpoint fingerprint differ")
    with manifest_path.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = [r for r in csv.DictReader(stream) if r["split"] == source_split]
    explicit_image = os.environ.get("CAD_VIT_AUDIT_IMAGE")
    if explicit_image:
        selected = [Path(explicit_image)]
    else:
        # Predefined deterministic selection: one image per actual grade, not selected by heatmap or prediction.
        selected = []
        for grade in range(5):
            group = [r for r in rows if int(r["label"]) == grade]
            if not group:
                print("Missing grade in selected split:", grade)
                continue
            row = min(group, key=lambda r: (hashlib.sha256(("42:" + r["image_id"]).encode()).hexdigest(), r["image_id"]))
            selected.append(dataset_root / row["image_path"])
    if not selected:
        raise ValueError("No verified non-test candidate images")
    device = torch.device(os.environ.get("CAD_VIT_GRADCAM_DEVICE", "cpu"))
    model = model.to(device).eval()
    for image_path in selected:
        checked_path, row = guarded_non_test_image(image_path, dataset_root, manifest_path, source_split)
        with Image.open(checked_path) as image:
            pixels = preprocess_fundus(image)
        tensor = torch.from_numpy(np.array(pixels, copy=True)).permute(2, 0, 1).float()[None].to(device) / 255
        result = gradcam(model, tensor)
        heatmap = result["heatmap"]
        if not torch.isfinite(heatmap).all():
            raise RuntimeError("Non-finite heatmap")
        print("image_id", row["image_id"], "split", source_split, "label", row["label"],
              "prediction", result["predicted_class"], "target", result["target_class"],
              "confidence", result["confidence"], "heatmap range", (float(heatmap.min()), float(heatmap.max())))
        if float(heatmap.max()) == float(heatmap.min()):
            print("WARNING: constant heatmap; no spatial attribution demonstrated")
        print("Provenance:", provenance)
        figure = plot_gradcam(pixels, result, int(row["label"]), provenance)
        display(figure)
        if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
            stem = "gradcam_" + "".join(ch if ch.isalnum() else "_" for ch in row["image_id"])
            print("Saved", save_campaign_figure(figure, ARTIFACT_ROOT, campaign, variant, stem))
        plt.close(figure)
else:
    print("PENDING real Grad-CAM: set CAD_VIT_RUN_GRADCAM=1 only after verifying existing artifacts")

**Eksekusi real-image.** Jalankan notebook di runtime yang memiliki dataset dan checkpoint asli. Set `CAD_VIT_ARTIFACT_ROOT` ke direktori artefak eksternal yang sudah ada; `CAD_VIT_RUN_GRADCAM=1` mengaktifkan real-image. `CAD_VIT_DATA_ROOT` opsional, harus berupa root dataset APTOS yang cocok dengan path relatif manifest; `CAD_VIT_AUDIT_MANIFEST` opsional. Default `val`, satu contoh deterministik per grade (seed string 42), tanpa memilih hasil berdasarkan prediksi atau heatmap. `CAD_VIT_AUDIT_IMAGE` opsional untuk satu gambar train/val. `CAD_VIT_GRADCAM_DEVICE` default CPU; pilih CUDA hanya jika GPU tersedia dan disetujui. `CAD_VIT_SAVE_FIGURES=1` opsional untuk menyimpan PNG pada direktori figures kampanye yang sudah digunakan proyek; default tidak menulis PNG eksternal. Simpan output notebook hanya setelah eksekusi benar-benar berhasil dan cek izin publikasi citra. Jangan menimpa hasil historis. Visualisasi adalah atribusi model, bukan validasi klinis lesi. Sel real-image ini belum dijalankan pada laptop; panel statis transfer server di bawah dibahas berdasarkan metadata yang tersedia dan bukan output sel tersebut.

## Grad-CAM nyata DDR — validation split

Grad-CAM menghitung bobot kanal dari rata-rata spasial gradien skor target terhadap activation map, menjumlahkan peta aktivasi berbobot, menerapkan ReLU, lalu memperbesar dan menormalisasi peta untuk overlay. Di sini target kelas adalah **kelas prediksi** dan layer `blocks.5.attention` merupakan **implementation assumption**, karena Figure 8 paper tidak menetapkan layer target. Original RGB ditampilkan terpisah dari input V1 512×512; overlay pada panel keempat mengikuti koordinat input model. Panel berasal dari arsip transfer server, bukan keluaran sel sintetis di atas.

![DDR: original, input V1, Grad-CAM, dan overlay untuk satu validation image per grade](../report_figures/DDR_gradcam_validation.png)

[Metadata lima gambar](../report_figures/DDR_gradcam_metadata.json) melaporkan split validation, checkpoint terbaik epoch **69**, SHA-256 checkpoint `84e3272d3f10d9179fd2dd9e2799b640962e1185981856f87bcb3876b9deace0`, manifest `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1`, serta `constant_heatmap=false` pada semua baris. Hash source model/preprocessing/protokol cocok dengan [config DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/config.json). Checkpoint dan manifest biner tidak tersedia lokal; hash dan identitas tiap gambar berasal dari metadata server, bukan verifikasi ulang lokal.

| Validation image ID | True→pred grade | Confidence | Target class |
| --- | ---: | ---: | ---: |
| 20170323130911479.jpg | 0→0 | 0.953 | 0 |
| 007-1758-100.jpg | 1→2 | 1.000 | 2 |
| 007-5096-300.jpg | 2→0 | 0.701 | 0 |
| 20170522144205431.jpg | 3→2 | 1.000 | 2 |
| 007-8689-605.jpg | 4→4 | 1.000 | 4 |

**Pembacaan tiap baris.** Grade 0 diprediksi benar; aktivasi tersebar di pembuluh dan tepi, tidak terkunci pada satu struktur. Grade 1 diprediksi 2 dengan confidence hampir 1; hotspot terdapat di beberapa area retina dan tepi atas, sehingga angka confidence tinggi tidak berarti interpretasi terlokalisasi. Grade 2 diprediksi 0; aktivasi lebih kuat pada sisi kiri dan bagian bawah, sedangkan area lain relatif rendah. Grade 3 diprediksi 2 dengan confidence hampir 1; hotspot tersebar di sekitar pembuluh dan tepi. Grade 4 diprediksi benar, tetapi citra asal sangat gelap dan heatmap luas pada bagian atas; pola ini sulit ditafsirkan sebagai bukti area penyakit tertentu. Pada beberapa baris hotspot juga mencapai pinggir/background. Tidak ada anotasi lesi, uji stabilitas lintas layer, atau kalibrasi confidence; peta ini **tidak membuktikan lokalisasi lesi maupun alasan klinis prediksi**.

## Grad-CAM nyata APTOS2019 — validation split, lokal saja

![APTOS2019 lokal: original, V1, Grad-CAM, dan overlay validation grade 0–4](../report_figures/APTOS2019_gradcam_validation.png)

[Metadata lima gambar](../report_figures/APTOS2019_gradcam_metadata.json) mencatat split validation, target kelas prediksi, layer `blocks.5.attention`, checkpoint pilihan validation epoch **24** (SHA-256 `6db01fbb840f4ed9641f9d8d3b9ac90e98d3b0772d57101be56aaabc60e05601`), dan manifest `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854`. Hash source model, preprocessing, dan protokol cocok dengan [config APTOS](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/config.json); `constant_heatmap=false` pada kelima baris. Checkpoint, manifest, dan citra asli tidak tersedia lokal untuk verifikasi ulang. Pilihan layer tetap *implementation assumption* terhadap Figure 8 paper.

| Validation image ID | True→pred grade | Confidence | Target class |
| --- | ---: | ---: | ---: |
| `0423237770a7` | 0→0 | 0,954 | 0 |
| `c96f743915b5` | 1→2 | 0,548 | 2 |
| `1594ca6c30d3` | 2→3 | 0,529 | 3 |
| `b960142a8de7` | 3→3 | 0,632 | 3 |
| `11242a67122d` | 4→2 | 0,804 | 2 |

**Pembacaan tiap baris.** Grade 0 diprediksi benar; hotspot terlihat pada sebagian perifer/bagian sekitar diskus, bukan satu titik tunggal. Grade 1 diprediksi 2 dengan aktivasi yang lebih menonjol di bagian atas/perifer. Grade 2 diprediksi 3 dan grade 3 diprediksi benar; keduanya menunjukkan beberapa area aktif yang tersebar, bukan satu lokalisasi lesi yang terverifikasi. Grade 4 diprediksi 2 dengan confidence **0,804**; aktivasi terlihat di bagian tengah dan tepi. Peta berasal dari lima gambar validation terpilih, bukan sampel untuk memperkirakan akurasi. Tanpa anotasi lesi, uji kestabilan lintas layer, atau audit ahli, Grad-CAM ini **tidak membuktikan alasan klinis prediksi**.

APTOS hanya ditampilkan pada salinan lokal yang memiliki file dengan hak akses; file diabaikan Git hingga izin berbagi dikonfirmasi. Tautan Markdown dapat dirender tanpa kernel secara lokal, tetapi tidak di GitHub/clone baru. [Status transfer](../report_figures/README.md#aptos2019-local-only-visualization-transfer). Hash ZIP transfer yang cocok mendukung identitas byte arsip terhadap catatan server peneliti, bukan bukti mandiri eksekusi atau lineage checkpoint. Source lokal kini memuat perbaikan alignment `plot_gradcam` dari `4e0f46f` dan guard validation APTOS dari `e14f3fe`. Sel real-image yang disinkronkan tetap belum dijalankan di laptop, dan tidak diklaim sebagai kode pembuat panel transfer server. Lisensi gambar DDR: [CC BY-NC-SA 4.0](../report_figures/README.md#ddr-real-image-visualization-transfer).